In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

In [ ]:
# 그래프 시각화를 위한 폰트 설정
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
font_list[:10]  # 일부만 출력
font_path='C:\\Windows\\Fonts'
mpl.rcParams['font.family'] = 'Malgun Gothic'

In [ ]:
# 데이터프레임을 불러오거나 생성합니다.
cluster_df = pd.read_excel('../P1_Data/02.Pretreatment/clustereData_sihueng.xlsx')
cluster_df

In [ ]:
# 문자열 열을 제외
numeric_columns = cluster_df.select_dtypes(include=['number'])
# 클러스터 개수 범위 설정
n_clusters = range(1, 10)
inertia = []  # 관성 값 저장

# 각 클러스터 개수에 대한 K-Means 모델을 학습하고 관성 값을 저장
for k in n_clusters:
    kmeans = KMeans(n_clusters=k)
    kmeans.fit(numeric_columns)
    inertia.append(kmeans.inertia_)

# 관성 값 그래프 그리기
plt.plot(n_clusters, inertia, marker='o')
plt.xlabel('클러스터 개수')
plt.ylabel('클러스터 내의 관성')
plt.title('Elbow Method')
# 그래프를 이미지 파일로 저장
plt.savefig('../P3_Result/02.PNG/클러스터개수.png', format='png')
plt.show()

In [ ]:
cluster_df_stand = cluster_df.copy()
# 데이터를 표준화
scaler = StandardScaler()
cluster_df_stand.iloc[:,1:] = scaler.fit_transform(cluster_df_stand.iloc[:,1:] )
cluster_df_stand = cluster_df_stand.round(5)


cluster_scale_positive = cluster_df_stand.copy()
# 숫자형 컬럼만 선택
numeric_columns = cluster_scale_positive.select_dtypes(include=['number'])

# 각 숫자형 컬럼별 최솟값 출력
min_values = numeric_columns.min()
min_values

In [ ]:
cluster_scale_positive['주민등록인구'] = cluster_scale_positive['주민등록인구'] + 2.00043
cluster_scale_positive['유동인구_비율'] = cluster_scale_positive['유동인구_비율'] + 1.02408
cluster_scale_positive['주중_집중도'] = cluster_scale_positive['주중_집중도'] + 2.42574
cluster_scale_positive['주민등록인구_60세이상_비율'] = cluster_scale_positive['주민등록인구_60세이상_비율'] + 1.40863
cluster_scale_positive['카드_월사용액'] = cluster_scale_positive['카드_월사용액'] + 1.34258
cluster_scale_positive['카드_주중비율'] = cluster_scale_positive['카드_주중비율'] + 1.50735
cluster_scale_positive['카드_증가율'] = cluster_scale_positive['카드_증가율'] + 2.67063
#cluster_scale_positive.to_csv('../P3_Result/01.CSV/3.유형화/시흥시_평균포함_표준화.csv',index=False,encoding='cp949')
cluster_scale_positive

In [ ]:

cluster_df_stand = cluster_df_stand[cluster_df_stand['행정동명'] != '평균']
scaled_data = cluster_df_stand.set_index('행정동명')

k = 4  # 클러스터 수 설정

kmeans = KMeans(n_clusters=k, random_state=0)
kmeans.fit(scaled_data)

# 클러스터 결과를 확인
cluster_labels = kmeans.labels_
cluster_df_stand['클러스터'] = cluster_labels
cluster_df_stand

In [ ]:
# '행정동명'과 '클러스터'를 제외한 열 선택
columns_to_standardize = cluster_df_stand.columns.difference(['행정동명', '클러스터'])

# 데이터프레임에서 선택한 열 추출
subset = cluster_df_stand[columns_to_standardize]

# 표준화된 데이터를 데이터프레임으로 변환
scaled_df = pd.DataFrame(data=subset, columns=columns_to_standardize)

# '행정동명'과 '클러스터' 열을 가장 앞에 추가
scaled_df.insert(0, '클러스터', cluster_df_stand['클러스터'])
scaled_df.insert(0, '행정동명', cluster_df_stand['행정동명'])
scaled_df

In [ ]:
# 번호에 따라 행정동을 그룹화하고 해당 행정동을 합친 새로운 열 생성
grouped = scaled_df.groupby('클러스터')['행정동명'].apply(lambda x: ','.join(x)).reset_index()
# 열 이름 변경
grouped.columns = ['클러스터', '해당행정동']
scaled_df = pd.merge(scaled_df,grouped,on='클러스터',how='inner')
#scaled_df.to_csv('../P3_Result/01.CSV/3.유형화/시흥시_클러스터링결과_표준화.csv',index=False,encoding='cp949')
scaled_df

In [ ]:
scaled_df_g = scaled_df.groupby(['클러스터','해당행정동'])[[ '유동인구_비율', '주민등록인구', '주민등록인구_60세이상_비율', '주중_집중도','카드_월사용액', '카드_주중비율', '카드_증가율']].mean().round(1).reset_index()
scaled_df_g['유동인구_비율'] = scaled_df_g['유동인구_비율'] + 0.4
scaled_df_g['주민등록인구'] = scaled_df_g['주민등록인구'] + 1.5
scaled_df_g['주민등록인구_60세이상_비율'] = scaled_df_g['주민등록인구_60세이상_비율'] + 0.5
scaled_df_g['주중_집중도'] = scaled_df_g['주중_집중도'] + 0.4
scaled_df_g['카드_월사용액'] = scaled_df_g['카드_월사용액'] + 1.3
scaled_df_g['카드_주중비율'] = scaled_df_g['카드_주중비율'] + 0.7
scaled_df_g['카드_증가율'] = scaled_df_g['카드_증가율'] + 1.5
#scaled_df_g.to_csv('../P3_Result/01.CSV/3.유형화/시흥시_클러스터링결과_유형별평균.csv',index=False,encoding='cp949')
scaled_df_g

In [ ]:
hjd_cluster = cluster_df_stand[['행정동명','클러스터']]
cluster_df_mg = pd.merge(cluster_df,hjd_cluster,on='행정동명')
#cluster_df_mg.to_csv('../P3_Result/01.CSV/3.유형화/시흥시_행정동별_클러스터유형.csv',index=False,encoding='cp949')
cluster_df_mg

In [ ]:
cluster_df_mg_scale = pd.merge(cluster_scale_positive,hjd_cluster,on='행정동명')
cluster_df_mg_scale.to_csv('../P3_Result/01.CSV/3.유형화/시흥시_행정동별_클러스터유형(표준화).csv',index=False,encoding='cp949')
cluster_df_mg_scale